# Retinal OCT classification with DenseNet121 and federated learning
**Fresh notebook — do not paste into the ODIR notebook.**

Dataset: Kermany OCT2017. Four mutually exclusive **image labels**: CNV (choroidal neovascularization), DME (diabetic macular edema), DRUSEN, NORMAL. This is OCT cross-sectional imaging, not ODIR fundus photography; it does not diagnose the eight ODIR categories or multiple simultaneous diseases. Suggested revised case-study title: **Federated DenseNet121 for Four-Class Retinal OCT Classification**.

The requested goal is **test accuracy > 0.85 AND precision > 0.85 in EACH class**. This notebook measures that exact condition honestly; it does not guarantee it. All images receive a prediction (no abstention or threshold tricks). No trained checkpoint or invented results are included.

### Run instructions
1. Upload this notebook to a NEW Google Colab notebook. Select Runtime → Change runtime type → GPU.
2. Run cells in order. Accept Drive mounting. Download/extraction can take time: the Kaggle archive is several GB, not a tiny dataset. Leave enough disk space. No old ODIR files are needed.
3. A 3-client, single-GPU FedAvg simulation is the default. Clients train sequentially; they are synthetic partitions, not real hospitals. No privacy guarantee or secure aggregation is implemented.
4. If training says PAUSED, rerun **Cell 6**. After a runtime reset, rerun cells 1 onward with the SAME configuration. Checkpoints persist in Drive; the local dataset/cache must be downloaded/rebuilt if the VM disappeared.
5. Cell 7 evaluates validation and writes the portable model bundle. **Cell 8 is final test evaluation:** run it only after training is finished and you accept the configuration. Running all cells follows that sequence automatically; a paused run blocks evaluation.
6. Do not tune the model after examining the final test and then call that same test an untouched evaluation.

### Split safeguards and compute budget
Patient IDs are parsed from the documented filename pattern `CLASS-patientID-imageNumber.jpeg`. Test patients are removed from the development pool before sampling. Development images sharing an exact decoded image with test are also excluded by connected patient group. Validation and simulated clients are group-disjoint, including duplicate-linked patients. Near duplicates are NOT exhaustively detected.

The existing tiny `val` folder, when present, is merged into the development pool; a new grouped validation split is formed there. The provided test folder is kept for final evaluation. All exclusions are logged. Therefore results are a **patient-audited derivative protocol**, not necessarily directly comparable to published OCT2017 scores. Earlier OCT dataset versions have documented overlap concerns.

To fit a free-Colab experiment, the default development pool is capped at 12 images per patient and 6,000 per class, before a roughly 80/20 train/validation split. Test images are not class-capped. Sampling is fixed before training, never selected by model performance. Actual counts are printed. This is a compute-limited subset, not full-data training; set caps to 0 BEFORE a new run for full-data experiments. All training data stays on this machine in the simulation.

In [ ]:
# CELL 1 — setup and configuration
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','kagglehub','tqdm'])
from google.colab import drive
from pathlib import Path
import os, json, random, re, hashlib, math, time, shutil
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchvision import transforms as T
from torchvision.models import densenet121,DenseNet121_Weights
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score,precision_recall_fscore_support,
    classification_report,confusion_matrix,ConfusionMatrixDisplay,roc_auc_score)
import matplotlib.pyplot as plt

drive.mount('/content/drive')
CFG=dict(seed=42,image_size=224,batch_size=16,accumulation=2,workers=2,
         clients=3,rounds=16,warmup_rounds=1,local_epochs=1,
         backbone_lr=1e-4,head_lr=5e-4,weight_decay=1e-4,
         label_smoothing=0.03,patience=5,max_per_patient=12,
         max_dev_per_class=6000,session_hours=2.5)
RUN_NAME='oct_densenet121_fedavg_v1'
OUT=Path('/content/drive/MyDrive/Retinal_OCT')/RUN_NAME
OUT.mkdir(parents=True,exist_ok=True)
DATA_ROOT='' # blank downloads Kaggle; otherwise path to extracted OCT dataset
CLASSES=['CNV','DME','DRUSEN','NORMAL']
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type=='cuda','Choose a GPU runtime before downloading/training.'
print('GPU:',torch.cuda.get_device_name(), 'Output:',OUT)
def seed_all(s):
    random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.cuda.manual_seed_all(s)
seed_all(CFG['seed'])
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
# Never independently upgrade torch/torchvision in Colab.

In [ ]:
# CELL 2 — download, robust directory discovery, and patient IDs
if not DATA_ROOT:
    import kagglehub
    assert shutil.disk_usage('/content').free>25*1024**3, 'Need at least 25 GiB free before download/extraction.'
    DATA_ROOT=kagglehub.dataset_download('paultimothymooney/kermany2018')
ROOT=Path(DATA_ROOT)
# Handles OCT2017 folders with trailing spaces and nested copies.
candidates=[]
for p in [ROOT]+list(ROOT.rglob('*')):
    if p.is_dir() and (p/'train').is_dir() and (p/'test').is_dir():
        if all((p/'train'/c).is_dir() and (p/'test'/c).is_dir() for c in CLASSES): candidates.append(p)
assert candidates, 'Could not find train/test folders with CNV, DME, DRUSEN, NORMAL. Set DATA_ROOT to the extracted dataset.'
# Repackaged archives may contain nested duplicate trees. Select one deterministic tree.
OCT_ROOT=sorted(candidates,key=lambda p:(len(p.parts),str(p)))[0]
print('Using:',OCT_ROOT)
pattern=re.compile(r'^(CNV|DME|DRUSEN|NORMAL)-(\d+)-(\d+)$',re.I)
records=[];bad=[]
for split in ['train','val','test']:
    for k,c in enumerate(CLASSES):
        folder=OCT_ROOT/split/c
        if not folder.exists():continue
        for p in sorted(folder.rglob('*')):
            if p.suffix.lower() not in ['.jpeg','.jpg','.png']:continue
            if p.name.startswith('._'):continue
            m=pattern.fullmatch(p.stem)
            if not m or m.group(1).upper()!=c:
                bad.append(str(p));continue
            # Do NOT include class in patient ID: one person may have several labels/scans.
            records.append(dict(path=str(p),rel=str(p.relative_to(OCT_ROOT)),
                patient=str(int(m.group(2))),label=k,source=split))
assert not bad, f'Unrecognized patient filenames; do not use an image-random split. Examples: {bad[:5]}'
raw=pd.DataFrame(records).sort_values('rel').reset_index(drop=True)
assert set(raw.label)==set(range(4)) and (raw.source=='test').any()
test_patients=set(raw.loc[raw.source=='test','patient'])
overlap=raw[(raw.source!='test')&raw.patient.isin(test_patients)]
overlap.to_csv(OUT/'excluded_test_patient_overlap.csv',index=False)
dev=raw[(raw.source!='test')&~raw.patient.isin(test_patients)].copy()
test=raw[raw.source=='test'].copy()
print('Removed development images belonging to test patients:',len(overlap))
# Fixed random selection before any model sees the data. No test subsampling.
rng=np.random.default_rng(CFG['seed'])
dev['selection_order']=rng.random(len(dev))
dev=dev.sort_values('selection_order')
if CFG['max_per_patient']:
    dev=dev.groupby('patient',sort=False).head(CFG['max_per_patient'])
if CFG['max_dev_per_class']:
    dev=dev.groupby('label',sort=False).head(CFG['max_dev_per_class'])
dev=dev.drop(columns='selection_order')
selected=pd.concat([dev,test],ignore_index=True).sort_values('rel').reset_index(drop=True)
assert selected.groupby('label').patient.nunique().min()>=10, 'Insufficient patients; inspect downloaded data.'
print('Selected images before duplicate audit:',len(selected))

In [ ]:
# CELL 3 — concurrent image audit/cache, grouped splits and client allocation
CACHE=Path('/content/oct_cache')/str(CFG['image_size'])
CACHE.mkdir(parents=True,exist_ok=True)
def preprocess(im,size):
    # Preserve the whole OCT scan; no random cropping or vertical flipping.
    im=ImageOps.exif_transpose(im).convert('RGB')
    return im.resize((size,size),Image.Resampling.BILINEAR)
def cache_image(row):
    try:
        with Image.open(row.path) as im:
            im=ImageOps.exif_transpose(im).convert('RGB')
            digest=hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
            # Per-source filename prevents two duplicate images racing to write one cache file.
            dest=CACHE/(hashlib.sha256(row.rel.encode()).hexdigest()+'.png')
            if not dest.exists():
                tmp=dest.with_suffix('.tmp')
                preprocess(im,CFG['image_size']).save(tmp,format='PNG',compress_level=1)
                os.replace(tmp,dest)
        return dict(rel=row.rel,digest=digest,cached=str(dest),error='')
    except Exception as e:return dict(rel=row.rel,digest='',cached='',error=str(e))
with ThreadPoolExecutor(max_workers=4) as pool:
    audit=list(tqdm(pool.map(cache_image,selected.itertuples(index=False)),total=len(selected),desc='Audit and resize'))
audit=pd.DataFrame(audit)
audit[audit.error!=''].to_csv(OUT/'corrupt_images.csv',index=False)
assert (audit.error=='').all(), 'Corrupt files found. Inspect corrupt_images.csv; repair download before proceeding.'
df=selected.merge(audit,on='rel',validate='one_to_one')
# Same image assigned conflicting labels must be reviewed, not silently relabeled.
# Detect identical images assigned different labels.
label_counts = df.groupby("digest")["label"].nunique()
conflicting_hashes = set(label_counts[label_counts > 1].index)

conflict_rows = df[df["digest"].isin(conflicting_hashes)].copy()
conflict_rows["class_name"] = conflict_rows["label"].map(
    dict(enumerate(CLASSES))
)

conflict_rows.to_csv(
    OUT / "conflicting_image_labels.csv",
    index=False
)

print("Conflicting image contents:", len(conflicting_hashes))
print("Affected image entries:", len(conflict_rows))

if len(conflict_rows):
    display(
        conflict_rows[
            ["rel", "patient", "class_name", "source", "digest"]
        ].head(20)
    )

# Check whether the TEST SET itself contains contradictory labels.
test_rows = df[df["source"] == "test"]
test_label_counts = test_rows.groupby("digest")["label"].nunique()
test_conflicts = test_label_counts[test_label_counts > 1]

if len(test_conflicts):
    raise ValueError(
        "Conflicting labels exist within the test set itself. "
        "Do not remove or relabel test images to improve the score. "
        "Share conflicting_image_labels.csv so we can inspect this."
    )

# Quarantine development patients involved in any label conflict.
# Keep test images unchanged; this does not assert that their labels
# are correct, only preserves the predefined evaluation cohort.
affected_dev_patients = set(
    conflict_rows.loc[
        conflict_rows["source"] != "test", "patient"
    ]
)

exclude = (
    (df["source"] != "test")
    & df["patient"].isin(affected_dev_patients)
)

df.loc[exclude].to_csv(
    OUT / "excluded_label_conflict_patients.csv",
    index=False
)

print("Development patients excluded:", len(affected_dev_patients))
print("Development images excluded:", int(exclude.sum()))

df = df.loc[~exclude].copy().reset_index(drop=True)

# Verify that conflicting labels have been resolved by exclusion.
remaining = df.groupby("digest")["label"].nunique()
assert (remaining <= 1).all(), "Unresolved label conflicts remain."

# Connect patient IDs when exact image content is shared.
parent={p:p for p in df.patient.unique()}
def find(p):
    while parent[p]!=p:
        parent[p]=parent[parent[p]];p=parent[p]
    return p
seen={}
for row in df.itertuples():
    if row.digest in seen:parent[find(row.patient)]=find(seen[row.digest])
    else:seen[row.digest]=row.patient
df['group']=df.patient.map(find)
test_groups=set(df.loc[df.source=='test','group'])
exclude=(df.source!='test')&df.group.isin(test_groups)
df[exclude].to_csv(OUT/'excluded_duplicate_linked_test_groups.csv',index=False)
df=df[~exclude].reset_index(drop=True)
# Deduplicate development images, retaining test cohort for transparent reporting.
dev=df[df.source!='test'].drop_duplicates('digest').reset_index(drop=True)
test=df[df.source=='test'].reset_index(drop=True)
fold=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=CFG['seed'])
ti,vi=next(fold.split(dev,dev.label,dev.group))
dev['split']='train';dev.loc[vi,'split']='val';test['split']='test'
df=pd.concat([dev,test],ignore_index=True)
df['client']=-1
train_indices=df.index[df.split=='train'].to_numpy()
train=df.loc[train_indices]
if CFG['clients']==1:df.loc[train_indices,'client']=0
else:
    cf=StratifiedGroupKFold(n_splits=CFG['clients'],shuffle=True,random_state=CFG['seed']+1)
    for c,(_,ix) in enumerate(cf.split(train,train.label,train.group)):
        df.loc[train_indices[ix],'client']=c
for a,b in [('train','val'),('train','test'),('val','test')]:
    da,db=df[df.split==a],df[df.split==b]
    assert set(da.patient).isdisjoint(db.patient)
    assert set(da.group).isdisjoint(db.group)
    assert set(da.digest).isdisjoint(db.digest)
assert df[df.split=='train'].groupby('group').client.nunique().max()==1
for s in ['train','val','test']:
    assert set(df.loc[df.split==s,'label'])==set(range(4)),f'Missing class in {s}'
for c in range(CFG['clients']):
    assert set(df.loc[df.client==c,'label'])==set(range(4)),f'Client {c} lacks a class; inspect data.'
counts=pd.crosstab(df.split,df.label).rename(columns=dict(enumerate(CLASSES)))
display(counts)
counts.to_csv(OUT/'split_class_counts.csv')
client_counts=pd.crosstab(df[df.split=='train'].client,df[df.split=='train'].label).rename(columns=dict(enumerate(CLASSES)))
display(client_counts);client_counts.to_csv(OUT/'client_class_counts.csv')
print('Patients per split:',df.groupby('split').patient.nunique().to_dict())
print('Duplicate image entries retained within test:',int(test.digest.duplicated().sum()))
manifest=df[['rel','patient','label','digest','group','split','client']].sort_values('rel').to_csv(index=False)
spec=dict(config={k:v for k,v in CFG.items() if k!='session_hours'},classes=CLASSES,
          manifest_sha256=hashlib.sha256(manifest.encode()).hexdigest(),architecture='oct_densenet121_v1')
if (OUT/'experiment.json').exists():
    assert json.loads((OUT/'experiment.json').read_text())==spec,'Settings/data changed. Restore them or use a NEW RUN_NAME.'
else:(OUT/'experiment.json').write_text(json.dumps(spec,indent=2))
(OUT/'manifest.csv').write_text(manifest)
(OUT/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
frames={s:df[df.split==s].reset_index(drop=True) for s in ['train','val','test']}

### Training protocol
DenseNet121 starts with ImageNet weights. One head-only warm-up round is followed by fine-tuning blocks 3 and 4 plus the head. BatchNorm running statistics and affine parameters remain frozen, a practical small-batch baseline (not FedBN). AdamW is reset at each local client update; a cosine learning-rate schedule is shared across rounds. Each round starts every client from the same global model. Updates are averaged by training image count. All clients participate.

Validation is a shared server-side research cohort. This simulation assumes access to data for constructing partitions and auditing leakage. There is no hospital-based privacy experiment. The one-client option (`clients=1`, NEW RUN_NAME) is a centralized reference, not federated learning.

Cross-entropy with mild label smoothing and softmax/argmax matches the single-label task. Checkpoint selection uses validation macro F1, not test accuracy. The >85% check includes every class; a high average cannot hide a weak category. Scores are not calibrated diagnostic probabilities.

In [ ]:
# CELL 4 — model, loaders and metrics
MEAN=[.485,.456,.406];STD=[.229,.224,.225]
train_tf=T.Compose([T.RandomHorizontalFlip(),T.RandomRotation(5,interpolation=T.InterpolationMode.BILINEAR),
                    T.ColorJitter(brightness=.08,contrast=.08),T.ToTensor(),T.Normalize(MEAN,STD)])
eval_tf=T.Compose([T.ToTensor(),T.Normalize(MEAN,STD)])
class OCTDataset(Dataset):
    def __init__(self,frame,training=False):self.f=frame.reset_index(drop=True);self.tf=train_tf if training else eval_tf
    def __len__(self):return len(self.f)
    def __getitem__(self,i):
        row=self.f.iloc[i]
        with Image.open(row.cached) as im:x=self.tf(im.convert('RGB'))
        return x,int(row.label)
def get_loader(frame,training=False):
    return DataLoader(OCTDataset(frame,training),batch_size=CFG['batch_size'],shuffle=training,
                      num_workers=CFG['workers'],pin_memory=True,drop_last=False)
class OCTNet(nn.Module):
    def __init__(self,pretrained=False):
        super().__init__()
        self.net=densenet121(weights=DenseNet121_Weights.DEFAULT if pretrained else None)
        self.net.classifier=nn.Sequential(nn.Dropout(.25),nn.Linear(1024,4))
    def forward(self,x):return self.net(x)
    def stage(self,warmup):
        for n,p in self.net.features.named_parameters():
            p.requires_grad=(not warmup and n.startswith(('denseblock3','transition3','denseblock4','norm5')))
        for m in self.net.features.modules():
            if isinstance(m,nn.BatchNorm2d):
                for p in m.parameters():p.requires_grad=False
    def train(self,mode=True):
        super().train(mode)
        for m in self.net.features.modules():
            if isinstance(m,nn.BatchNorm2d):m.eval()
        return self

def cpu_state(m):return {k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
def save_atomic(obj,path):
    tmp=Path(str(path)+'.tmp');torch.save(obj,tmp);os.replace(tmp,path)
def load_own(path):
    # Only load files generated by this notebook; pickle checkpoints from others are unsafe.
    return torch.load(path,map_location='cpu',weights_only=False)
@torch.inference_mode()
def probabilities(net,frame):
    net.eval();ys=[];ps=[]
    for x,y in get_loader(frame):
        with torch.autocast(device_type='cuda',dtype=torch.float16):logits=net(x.to(DEVICE,non_blocking=True))
        ys.append(y.numpy());ps.append(logits.float().softmax(1).cpu().numpy())
    return np.concatenate(ys),np.concatenate(ps)
def measure(y,p):
    pred=p.argmax(1)
    precision,recall,f1,support=precision_recall_fscore_support(y,pred,labels=range(4),zero_division=0)
    summary=dict(accuracy=float(accuracy_score(y,pred)),macro_precision=float(precision.mean()),
                 macro_recall=float(recall.mean()),macro_F1=float(f1.mean()),
                 minimum_class_precision=float(precision.min()),
                 target_met=bool(accuracy_score(y,pred)>.85 and (precision>.85).all()))
    table=pd.DataFrame(dict(class_name=CLASSES,precision=precision,recall=recall,F1=f1,support=support,
                            predicted_count=np.bincount(pred,minlength=4)))
    return summary,table

In [ ]:
# CELL 5 — one local client update
loss_fn=nn.CrossEntropyLoss(label_smoothing=CFG['label_smoothing'])
def local_update(net,frame,r,c):
    seed_all(CFG['seed']+1000*r+c)
    net.stage(r<CFG['warmup_rounds']);net.train()
    t=max(0,r-CFG['warmup_rounds'])/max(1,CFG['rounds']-CFG['warmup_rounds']-1)
    scale=.1+.9*(1+math.cos(math.pi*t))/2
    opt=torch.optim.AdamW([
        dict(params=[p for p in net.net.features.parameters() if p.requires_grad],lr=CFG['backbone_lr']*scale),
        dict(params=net.net.classifier.parameters(),lr=CFG['head_lr']*scale)],weight_decay=CFG['weight_decay'])
    scaler=torch.amp.GradScaler('cuda');total=0.;seen=0
    dl=get_loader(frame,True)
    for epoch in range(CFG['local_epochs']):
        opt.zero_grad(set_to_none=True);window=0
        for step,(x,y) in enumerate(tqdm(dl,desc=f'Round {r+1}/{CFG["rounds"]}, client {c+1}/{CFG["clients"]}',leave=False)):
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16):loss=loss_fn(net(x),y)
            assert torch.isfinite(loss),'Nonfinite loss; stop and inspect input/training settings.'
            scaler.scale(loss*len(y)).backward();window+=len(y)
            total+=loss.item()*len(y);seen+=len(y)
            if (step+1)%CFG['accumulation']==0 or step+1==len(dl):
                scaler.unscale_(opt)
                for param in net.parameters():
                    if param.grad is not None:param.grad.div_(window)
                nn.utils.clip_grad_norm_([p for p in net.parameters() if p.requires_grad],5.)
                scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True);window=0
    return total/seen

In [ ]:
# CELL 6 — federated training, resumable after every completed client
# Re-run THIS cell after a PAUSED message. Do not change settings to resume.
def train_federated():
    latest=OUT/'latest.pt'
    if latest.exists():
        ck=load_own(latest);assert ck['spec']==spec,'Checkpoint belongs to a different experiment.'
    else:
        seed_all(CFG['seed']);initial=OCTNet(True)
        ck=dict(spec=spec,global_state=cpu_state(initial),round=0,next_client=0,
                accumulator={},weight=0,losses=[],history=[],best=-1.,bad=0,done=False)
        del initial;save_atomic(ck,latest)
    net=OCTNet().to(DEVICE);started=time.monotonic()
    while ck['round']<CFG['rounds'] and not ck['done']:
        r=ck['round']
        for c in range(ck['next_client'],CFG['clients']):
            if time.monotonic()-started>CFG['session_hours']*3600:
                print('PAUSED at saved client boundary. Run Cell 6 again to continue.');return
            local=frames['train'][frames['train'].client==c].reset_index(drop=True)
            net.load_state_dict(ck['global_state'])
            loss=local_update(net,local,r,c);state=cpu_state(net);n=len(local)
            for k,v in state.items():
                if v.is_floating_point():
                    if k not in ck['accumulator']:ck['accumulator'][k]=torch.zeros_like(v,dtype=torch.float32)
                    ck['accumulator'][k].add_(v.float(),alpha=n)
                else:assert torch.equal(v,ck['global_state'][k])
            del state
            ck['weight']+=n;ck['losses'].append((n,loss));ck['next_client']=c+1
            save_atomic(ck,latest)
        for k,v in ck['global_state'].items():
            if v.is_floating_point():ck['global_state'][k]=(ck['accumulator'][k]/ck['weight']).to(v.dtype)
        net.load_state_dict(ck['global_state'])
        y,p=probabilities(net,frames['val']);metrics,_=measure(y,p)
        metrics.update(round=r+1,train_loss=sum(n*l for n,l in ck['losses'])/ck['weight'])
        ck['history'].append(metrics)
        if metrics['macro_F1']>ck['best']:
            ck['best']=metrics['macro_F1'];ck['bad']=0
            save_atomic(dict(model=ck['global_state'],spec=spec,round=r+1,validation=metrics),OUT/'best.pt')
        elif r>=CFG['warmup_rounds']:ck['bad']+=1
        ck['done']=ck['bad']>=CFG['patience'] or r+1>=CFG['rounds']
        ck.update(round=r+1,next_client=0,accumulator={},weight=0,losses=[])
        save_atomic(ck,latest)
        pd.DataFrame(ck['history']).to_csv(OUT/'training_history.csv',index=False)
        print(f'Round {r+1}: val accuracy={metrics["accuracy"]:.3f}, macro F1={metrics["macro_F1"]:.3f}, '
              f'min class precision={metrics["minimum_class_precision"]:.3f}, target={metrics["target_met"]}')
    print('FINISHED. Best validation macro F1:',ck['best'])
train_federated()

In [ ]:
# CELL 7 — fresh validation evaluation and model export (no stale model variables)
def selected_model():
    ck=load_own(OUT/'latest.pt')
    assert ck['spec']==spec
    assert ck['done'],'Training PAUSED, not finished. Re-run Cell 6, then this cell.'
    best=load_own(OUT/'best.pt');assert best['spec']==spec
    net=OCTNet().to(DEVICE);net.load_state_dict(best['model']);net.eval()
    return net,best
net,best=selected_model()
yv,pv=probabilities(net,frames['val'])
summary,table=measure(yv,pv)
assert abs(summary['macro_F1']-best['validation']['macro_F1'])<1e-6,'Checkpoint evaluation mismatch.'
display(table);print('VALIDATION:',json.dumps(summary,indent=2),'Best round:',best['round'])
table.to_csv(OUT/'validation_per_class.csv',index=False)
np.savez_compressed(OUT/'validation_predictions.npz',y=yv,p=pv,
                    ids=frames['val'].rel.to_numpy(dtype=str),patients=frames['val'].patient.to_numpy(dtype=str))
save_atomic(dict(model=best['model'],classes=CLASSES,image_size=CFG['image_size'],
                 mean=MEAN,std=STD,preprocess='full_image_resize_bilinear_rgb',
                 spec=spec,best_round=best['round']),OUT/'inference_bundle.pt')
h=pd.read_csv(OUT/'training_history.csv')
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(h['round'],h.train_loss);ax[0].set(title='Mean local training loss',xlabel='Round')
for col in ['accuracy','macro_F1','minimum_class_precision']:ax[1].plot(h['round'],h[col],label=col)
ax[1].axhline(.85,color='gray',linestyle='--');ax[1].set(xlabel='Round',title='Validation metrics');ax[1].legend()
fig.tight_layout();fig.savefig(OUT/'learning_curves.png',dpi=150);plt.show()
del net;torch.cuda.empty_cache()

### Final evaluation — read before running the next cell
The next cell evaluates **every held-out test image**, without confidence filtering or threshold optimization. It reloads the checkpoint and binds results to its SHA256 and dataset manifest, preventing old metrics from being shown as new ones. Metrics are image-level; patient/duplicate-group bootstrap intervals reflect correlated images. They do not include uncertainty across training seeds. A test set with few independent patients cannot establish general clinical performance.

Do not run this cell repeatedly to select training settings. If the target fails, report that honestly. Benchmark accuracy is not clinical validation. Your earlier ODIR test results are unrelated to this new OCT task and must not be pooled with it.

In [ ]:
# CELL 8 — final test; automatically loads the selected checkpoint
# No RUN_FINAL_TEST toggle and no dependency on an old in-memory model.
def final_test():
    net,best=selected_model()
    model_hash=hashlib.sha256((OUT/'best.pt').read_bytes()).hexdigest()
    key=dict(model_sha256=model_hash,manifest_sha256=spec['manifest_sha256'])
    result_file=OUT/'test_metrics.json'
    if result_file.exists():
        previous=json.loads(result_file.read_text())
        assert previous['evaluation_key']==key,'Saved test belongs to a different model. Use a separate experiment folder.'
        display(pd.read_csv(OUT/'test_per_class.csv'));print(json.dumps(previous,indent=2));return
    yt,pt=probabilities(net,frames['test']);metrics,table=measure(yt,pt)
    metrics.update(evaluation_key=key,best_round=best['round'],test_images=len(yt),
                   test_patients=int(frames['test'].patient.nunique()),
                   independent_groups=int(frames['test'].group.nunique()),
                   macro_ROC_AUC=float(roc_auc_score(yt,pt,multi_class='ovr',average='macro')))
    # Pre-specified cluster bootstrap; no test-driven model/threshold selection.
    groups=frames['test'].group.to_numpy();ug=np.unique(groups)
    rows={g:np.flatnonzero(groups==g) for g in ug};rng=np.random.default_rng(CFG['seed'])
    boot=[]
    for _ in range(300):
        ix=np.concatenate([rows[g] for g in rng.choice(ug,len(ug),replace=True)])
        if len(np.unique(yt[ix]))!=4:continue
        s,_=measure(yt[ix],pt[ix]);boot.append([s['accuracy'],s['macro_precision'],s['minimum_class_precision']])
    if boot:
        ci=np.percentile(boot,[2.5,97.5],axis=0)
        for i,name in enumerate(['accuracy','macro_precision','minimum_class_precision']):metrics[name+'_95CI']=ci[:,i].tolist()
    metrics['bootstrap_valid_replicates']=len(boot)
    table.to_csv(OUT/'test_per_class.csv',index=False)
    np.savez_compressed(OUT/'test_predictions.npz',y=yt,p=pt,
                        ids=frames['test'].rel.to_numpy(dtype=str),patients=frames['test'].patient.to_numpy(dtype=str))
    result_file.write_text(json.dumps(metrics,indent=2))
    display(table);print(json.dumps(metrics,indent=2))
    fig,ax=plt.subplots(figsize=(7,6))
    ConfusionMatrixDisplay(confusion_matrix(yt,pt.argmax(1),labels=range(4)),display_labels=CLASSES).plot(ax=ax,cmap='Blues',colorbar=False)
    fig.tight_layout();fig.savefig(OUT/'confusion_matrix.png',dpi=160);plt.show()
    print('TARGET MET on this test sample.' if metrics['target_met'] else 'TARGET NOT MET on this test sample.')
    print('Target requires accuracy > 0.85 AND precision > 0.85 in EVERY class. All images were evaluated.')
final_test()

In [ ]:
# CELL 9 — predict a new OCT scan (NOT a fundus photo)
@torch.inference_mode()
def predict_oct(image_path,bundle_path=OUT/'inference_bundle.pt'):
    b=load_own(bundle_path)
    model=OCTNet().to(DEVICE);model.load_state_dict(b['model']);model.eval()
    with Image.open(image_path) as im:
        prepared=preprocess(im,b['image_size'])
    transform=T.Compose([T.ToTensor(),T.Normalize(b['mean'],b['std'])])
    x=transform(prepared).unsqueeze(0).to(DEVICE)
    with torch.autocast(device_type='cuda',dtype=torch.float16):logits=model(x)
    scores=logits.float().softmax(1).cpu().numpy()[0]
    result=pd.DataFrame({'category':b['classes'],'score_uncalibrated':scores})
    print('Predicted OCT category:',b['classes'][int(scores.argmax())])
    display(result.sort_values('score_uncalibrated',ascending=False))
    print('Research classification, not a clinical diagnosis. Scores are not calibrated risk estimates.')
    return result
# Upload an OCT image using the Colab Files panel, then uncomment:
# result = predict_oct('/content/my_oct_scan.jpeg')

## Troubleshooting and reporting
- **GPU unavailable:** reconnect when Colab offers one; free access and session duration are not guaranteed. CPU is not a practical fallback for the full training run.
- **Out of GPU memory before training:** use batch_size=8, accumulation=4 in a NEW RUN_NAME. The image batch remains effectively 32. Do not change experiment settings mid-resume. Close other GPU models if you ran unrelated cells.
- **Dataset download requires authentication:** follow Kaggle's login instructions; you can run `import kagglehub; kagglehub.login()` interactively. Never put tokens in public notebooks. Alternatively extract the official OCT dataset into `/content` and set DATA_ROOT. The official source is linked below. Do not select the chest X-ray dataset in that archive.
- **Pause/disconnect:** latest.pt contains the round-start weights plus aggregation of completed clients. An interrupted client is repeated, not resumed mid-batch. This is intentional. The time budget is checked between clients and may be exceeded by one update.
- **Cache preprocessing:** this notebook resizes only the selected cohort and uses four threads. It does not require repeating ODIR's full-resolution fundus processing. Local cache is not persistent across VM deletion; Drive checkpoints are.
- **Checkpoint selection:** validation macro F1, fixed argmax rule. Reports distinguish accuracy, macro precision, minimum class precision and recall. No hidden threshold filtering, no dropped hard test cases, no use of test scores for early stopping.
- **Protocol limitations:** patient IDs derive from filenames. Exact decoded-content checks miss transformed/near duplicates. Development class caps change its distribution. Published OCT2017 scores may use different splits, releases or leakage handling; do not claim direct comparability.
- **Case-study claim:** this is a 4-class retinal OCT image classification experiment with simulated clients. It is NOT the original eight-label ODIR problem, a real hospital federation, a differential-private model, or a clinically validated diagnostic system. Report actual independent patient/group counts and per-class support with results.
- **Research target:** accuracy >85% and all four class precisions >85% is measured, not promised. Small supports and uncertainty matter even if the numerical target is met. Preserve failed runs in the study rather than reporting only favorable metrics.

## Sources
- [Dataset authors, Mendeley dataset and filename convention](https://data.mendeley.com/datasets/rscbjbr9sj)
- [Kaggle OCT2017 mirror](https://www.kaggle.com/datasets/paultimothymooney/kermany2018)
- [Kermany et al., Identifying Medical Diagnoses and Treatable Diseases by Image-Based Deep Learning](https://pubmed.ncbi.nlm.nih.gov/29474911/)
- [Tampu et al., Inflation of test accuracy due to data leakage in deep learning-based classification of OCT images](https://pmc.ncbi.nlm.nih.gov/articles/PMC9500039/)
- [Torchvision DenseNet121](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.densenet121.html)
- [FedAvg paper](https://arxiv.org/abs/1602.05629)
- [Colab resource limits](https://research.google.com/colaboratory/faq.html)

**Authoring validation:** code syntax and synthetic preprocessing/split/metric checks were performed. Full dataset download, GPU training and an actual >85% result were not executed in the authoring environment. The notebook contains no fabricated outputs.